# Bronze profiling notebook

Before mapping a new Bronze table to Silver, profile it: row count, distinct keys, null rate per column,
and the date range. This is the Spark version of the manual checks done before each Silver build
(duplicate keys, column existence, date ranges for reconciliation).

**Status:** designed for the next phase of the project; not yet run in production.

In [ ]:
table_name = "br_oracle_fusion_fscm_prjextractam_pjcbiccextractam_projectcostdistribution"
key_columns = "PjcCostDistLinesAllExpenditureItemId,PjcCostDistLinesAllLineNum"
date_column = "PjcCostDistLinesAllPrvdrGlDate"

In [ ]:
from pyspark.sql import functions as F

df = spark.read.table(f"bronze.{table_name}")
keys = key_columns.split(",")

total = df.count()
distinct_keys = df.select(*keys).distinct().count()
print(f"rows: {total:,}   distinct keys: {distinct_keys:,}   duplicate rows: {total - distinct_keys:,}")

nulls = df.select([(F.sum(F.col(c).isNull().cast("int")) / F.lit(total)).alias(c) for c in df.columns])
null_rate = (spark.createDataFrame([(c, float(nulls.first()[c])) for c in df.columns], "column string, null_rate double")
                  .orderBy(F.desc("null_rate")))
display(null_rate)

In [ ]:
# Month-by-month row counts: the same view used to reconcile Bronze with the source system.
monthly = (df.withColumn("month", F.date_format(F.col(date_column).cast("date"), "yyyy-MM"))
             .groupBy("month").count().orderBy("month"))
display(monthly)